# Self-Improving Agent Loop

This notebook implements a self-improving loop where:
1. Base agent attempts to answer questions
2. Failures are passed to the proposer to suggest skills
3. Skill generator creates the proposed skills
4. New mutations are evaluated and added to frontier if improved
5. Loop continues until threshold or max iterations

## 1. Imports & Setup

In [ ]:
from pathlib import Path
import os
import pandas as pd
from src.agent_profiles import Agent, base_agent_options, proposer_options, skill_generator_options, prompt_generator_options
from src.agent_profiles.base_agent import get_base_agent_options
from src.schemas import AgentResponse, ProposerResponse, ToolGeneratorResponse, PromptGeneratorResponse
from src.registry import ProgramManager, ProgramConfig
from src.agent_profiles.skill_generator import get_project_root
from src.evaluation import score_answer

# Load dataset
dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
csv_path = os.path.join(dataset_path, "officeqa.csv")
train_data = pd.read_csv(csv_path)
hard_data = train_data[train_data['difficulty'] == 'hard']
easy_data = train_data[train_data['difficulty'] == 'easy']

In [ ]:
# base_agent_options.model = 'sonnet'
# proposer_options.model = 'sonnet'
# skill_generator_options.model = 'sonnet'

: 

## 2. Configuration

In [7]:
# Loop configuration
MAX_ITERATIONS = 5
FRONTIER_SIZE = 3
NO_IMPROVEMENT_LIMIT = 5

# Sample questions for evaluation (use a subset for testing)
sample_questions = [
    (row.question, row.answer) 
    for _, row in hard_data.head(5).iterrows()
]

## 3. Initialize Agents & Manager

In [ ]:
# Create agents
base_agent = Agent(base_agent_options, AgentResponse)
proposer = Agent(proposer_options, ProposerResponse)
skill_generator = Agent(skill_generator_options, ToolGeneratorResponse)
prompt_generator = Agent(prompt_generator_options, PromptGeneratorResponse)

# Initialize program manager
manager = ProgramManager(cwd=get_project_root())

# Feedback history file
feedback_path = Path(get_project_root()) / ".claude" / "feedback_history.md"
# Updated to use .txt instead of .py
prompt_path = Path(get_project_root()) / "src" / "agent_profiles" / "base_agent" / "prompt.txt"

## 4. Helper Functions

In [ ]:
def get_best_parent(manager: ProgramManager) -> str:
    """Get best program from frontier, or 'base' if frontier is empty."""
    best = manager.get_best_from_frontier()
    return best if best else "base"


def evaluate_agent(agent: Agent, questions: list[tuple[str, str]]) -> float:
    """
    Evaluate agent on a set of questions.
    Returns accuracy score (0.0 to 1.0).
    
    TODO: Replace this stub with actual evaluation logic.
    """
    # Stub: return random score for now
    import random
    return random.uniform(0.3, 0.8)


def build_proposer_query(trace, answer: str, feedback_history: str) -> str:
    """Build the query for the proposer agent."""
    # Handle failed parses
    if trace.output is None:
        agent_answer = f"[PARSE FAILED: {trace.parse_error}]"
    else:
        agent_answer = trace.output.final_answer
    
    return f"""## Previous Attempts Feedback
{feedback_history}

## Current Attempt
{trace.summarize()}

Agent Answer: {agent_answer}
Ground Truth: {answer}"""


def build_skill_query(proposer_trace) -> str:
    """Build the query for the skill generator agent."""
    return f"""Proposed tool or skill (high level description): {proposer_trace.output.proposed_skill_or_prompt}

Justification: {proposer_trace.output.justification}"""

def build_prompt_query(proposer_trace, original_prompt: str) -> str:
    """Build the query for the prompt generator agent."""
    return f"""## Original Prompt
{original_prompt}

## Proposed Change
{proposer_trace.output.proposed_skill_or_prompt}

## Justification
{proposer_trace.output.justification}"""


def append_feedback(path: Path, iteration: str, skill: str, justification: str):
    """Append feedback entry to history file."""
    entry = f"""
## {iteration}
**Skill or Prompt**: {skill}
**Justification**: {justification}

"""
    with open(path, "a") as f:
        f.write(entry)


def read_feedback_history(path: Path) -> str:
    """Read feedback history or return default message."""
    if path.exists():
        return path.read_text()
    return "No previous attempts."


def update_prompt_file(file_path: Path, new_prompt: str):
    """
    Write the new prompt as plain text to prompt.txt.
    The Agent reads this file at runtime on each run().
    """
    file_path.write_text(new_prompt.strip())
    print(f"Updated {file_path}")

## 5. Create Base Program

In [ ]:
# Create base program if it doesn't exist
if "base" not in manager.list_programs():
    # Get current options (call the factory function)
    current_options = get_base_agent_options()
    
    base_config = ProgramConfig(
        name="base",
        parent=None,
        generation=0,
        system_prompt=current_options.system_prompt,
        allowed_tools=current_options.allowed_tools,
        output_format=current_options.output_format,
        metadata={}
    ).with_timestamp()
    
    manager.create_program("base", base_config)
    print(f"Created program/base")
else:
    print(f"program/base already exists")

# Evaluate and add base to frontier
manager.switch_to("base")
base_score = evaluate_agent(base_agent, sample_questions)
manager.update_frontier("base", base_score, max_size=FRONTIER_SIZE)
print(f"Base score: {base_score:.4f}")
print(f"Frontier: {manager.get_frontier()}")

## 6. Main Self-Improving Loop

In [ ]:
no_improvement_count = 0
iteration_count = 0

for i in range(MAX_ITERATIONS):
    iteration_count = i + 1
    print(f"\n{'='*50}")
    print(f"Iteration {iteration_count}")
    print(f"{'='*50}")
    
    # 1. Select best parent from frontier
    parent = get_best_parent(manager)
    manager.switch_to(parent)
    print(f"Selected parent: {parent}")
    
    # 2. Pick a sample question
    question, answer = sample_questions[i % len(sample_questions)]
    print(f"Question: {question[:80]}...")
    
    # 3. Run agent
    trace = await base_agent.run(question)
    print(f"Agent answer: {trace.output.final_answer[:80]}...")
    print(f"Ground truth: {answer[:80]}...")
    
    # 4. Check if correct (simple string match for now)
    is_correct = trace.output.final_answer.strip().lower() == answer.strip().lower()
    
    if is_correct:
        print("✓ Correct! Skipping mutation.")
        continue
    
    print("✗ Incorrect. Running proposer...")
    
    # 5. Run proposer to suggest skill
    feedback_history = read_feedback_history(feedback_path)
    proposer_query = build_proposer_query(trace, answer, feedback_history)
    proposer_trace = await proposer.run(proposer_query)
    
    prompt_or_skill_proposed = proposer_trace.output.optimize_prompt_or_skill
    proposed_skill_or_prompt = proposer_trace.output.proposed_skill_or_prompt
    justification = proposer_trace.output.justification

    print(f"Prompt or skill selected: {prompt_or_skill_proposed}")
    print(f"Proposed skill or prompt: {proposed_skill_or_prompt}")
    
    # 6. Create child program branch
    child_name = f"iter-{iteration_count}"
    parent_config = manager.get_current()
    original_prompt = parent_config.system_prompt
    child_config = parent_config.mutate(child_name)
    manager.create_program(child_name, child_config, parent=parent)
    print(f"Created child: {child_name}")
    
    # 7. Generate skill

    if prompt_or_skill_proposed == "prompt":
        prompt_query = build_prompt_query(proposer_trace, original_prompt)
        prompt_trace = await prompt_generator.run(prompt_query)
        update_prompt_file(prompt_path, prompt_trace.output.optimized_prompt)
        print(f"Generated prompt: {prompt_trace.output.optimized_prompt[:100]}...")
    else:
        skill_query = build_skill_query(proposer_trace)
        skill_trace = await skill_generator.run(skill_query)
        print(f"Generated skill: {skill_trace.output.generated_skill[:100]}...")

    
    # 8. Commit changes
    manager.commit(f"{child_name}: {proposed_skill_or_prompt[:50]}")
    
    # 9. Append to feedback history
    append_feedback(feedback_path, child_name, proposed_skill_or_prompt, justification)
    
    # 10. Evaluate child
    child_score = evaluate_agent(base_agent, sample_questions)
    print(f"Child score: {child_score:.4f}")
    
    # 11. Update frontier or discard
    added_to_frontier = manager.update_frontier(child_name, child_score, max_size=FRONTIER_SIZE)
    
    if added_to_frontier:
        print(f"✓ Added to frontier!")
        no_improvement_count = 0
    else:
        print(f"✗ Not good enough. Discarding...")
        manager.discard(child_name)
        no_improvement_count += 1
    
    # 12. Check early stopping
    if no_improvement_count >= NO_IMPROVEMENT_LIMIT:
        print(f"\nEarly stopping: {NO_IMPROVEMENT_LIMIT} iterations without improvement.")
        break
    
    # Print frontier status
    print(f"Current frontier: {manager.get_frontier_with_scores()}")


Iteration 1
Selected parent: base
Question: What were the total expenditures (in millions of nominal dollars) for U.S nation...
Agent answer: The total expenditures for U.S. national defense in the calendar year 1940 were ...
Ground truth: 2,602...
✗ Incorrect. Running proposer...
Proposed skill: {
  "name": "verify_numerical_data",
  "description": "Cross-validates numerical/statistical data by fetching from multiple authoritative sources and comparing values",
  "type": "tool",
  "implementation": "def verify_numerical_data(query: str, expected_range: Optional[tuple] = None) -> dict:\n    \"\"\"\n    Searches multiple authoritative sources for numerical data and returns:\n    - All found values with sources\n    - Consistency score across sources\n    - Potential reasons for discrepancies (fiscal vs calendar year, etc.)\n    - Recommended value with confidence level\n    \n    Args:\n        query: The numerical fact to verify (e.g., 'U.S. defense spending 1940')\n        expected_r

## 7. Results Summary

In [12]:
print(f"\n{'='*50}")
print("FINAL RESULTS")
print(f"{'='*50}")
print(f"Iterations completed: {iteration_count}")
print(f"\nFrontier (best agents):")
for name, score in manager.get_frontier_with_scores():
    print(f"  - {name}: {score:.4f}")

best = manager.get_best_from_frontier()
if best:
    print(f"\nBest agent: {best}")
    best_config = manager._read_config_from_branch(f"program/{best}")
    print(f"Generation: {best_config.generation}")
    print(f"Lineage: {manager.get_lineage(best)}")


FINAL RESULTS
Iterations completed: 5

Frontier (best agents):
  - iter-3: 0.7198
  - iter-2: 0.6376
  - iter-5: 0.5100

Best agent: iter-3
Generation: 3
Lineage: ['iter-3', 'iter-2', 'iter-1', 'base']


## 8. Cleanup (Optional)

In [ ]:
# Switch back to main
manager._git_checkout("main")
print(f"Switched to: {manager._git_current_branch()}")

# # Optionally discard all test branches
# for program in manager.list_programs():
#     manager.discard(program)
#     print(f"Discarded: {program}")

Switched to: main
Discarded: base
Discarded: iter-1
Discarded: iter-2
Discarded: iter-3
Discarded: iter-4
Discarded: iter-5
